# 📰 AI-Powered Fake News Detection Pipeline (From Scratch)
**Author:** Rohit Korpal (Roll No. 087996)


This notebook implements a complete end-to-end Machine Learning pipeline to classify news articles as **Real** or **Fake**. All core components—raw text preprocessing, tokenization, feature extraction (TF-IDF), model evaluation metrics, and classification models (K-Nearest Neighbors, Logistic Regression, Feedforward Neural Network, and Random Forest)—are built **entirely from scratch** using only `NumPy` and `Pandas` (without relying on libraries like `scikit-learn` or `nltk` for modeling).

### Pipeline Components:
1. **Exploratory Data Analysis (EDA)**: Distinguishing characteristics of True vs. Fake articles and uncovering target leakage.
2. **Text Preprocessing**: Normalization, URL/HTML removal, punctuation replacement, dateline/publisher signature purging, and stopword removal.
3. **Feature Engineering**: Custom TF-IDF vectorization with L2 normalization.
4. **Classifiers (From Scratch)**: K-Nearest Neighbors (with Cosine distance), Logistic Regression, Simple Neural Network, and Random Forest.
5. **Evaluation**: From-scratch calculations for Accuracy, Precision, Recall, F1 score, and Confusion Matrices.
6. **Analysis**: Comparative validation, training times, and loss convergence curves.


In [ ]:
import os
import time
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12


## Section 1: Data Exploration (EDA)
In this section, we load the dataset and perform basic Exploratory Data Analysis. The dataset consists of two CSV files located in the `dataset/` directory:
1. `True.csv`: Contains authentic news articles from Reuters.
2. `Fake.csv`: Contains fabricated news articles from various online sources.

Let's read these files, label them (`0` for Real, `1` for Fake), and explore their distribution and structure.


In [ ]:
def load_data(sample_size=2000):
    true_path = os.path.join("dataset", "True.csv")
    fake_path = os.path.join("dataset", "Fake.csv")
    if not os.path.exists(true_path) or not os.path.exists(fake_path):
        raise FileNotFoundError("True.csv and Fake.csv must be in the 'dataset' directory.")
        
    df_true = pd.read_csv(true_path)
    df_fake = pd.read_csv(fake_path)
    
    print(f"Found {len(df_true)} Real and {len(df_fake)} Fake articles.")
    
    # Label the datasets: 0 = Real, 1 = Fake
    df_true['label'] = 0
    df_fake['label'] = 1
    
    # Merge datasets
    df = pd.concat([df_true, df_fake], ignore_index=True)
    
    # Sample balanced sets if sample_size is smaller
    if sample_size and sample_size < len(df):
        half_sample = sample_size // 2
        df_true_sampled = df_true.sample(half_sample, random_state=42)
        df_fake_sampled = df_fake.sample(half_sample, random_state=42)
        df_sampled = pd.concat([df_true_sampled, df_fake_sampled], ignore_index=True)
        # Shuffle
        df_sampled = df_sampled.sample(frac=1, random_state=42).reset_index(drop=True)
        return df_sampled
        
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

# Load a sample dataset for demonstration
df = load_data(sample_size=2000)
df.head()


### EDA - Text Length Analysis & Target Leakage Check
Let's analyze the word length distributions of real vs. fake news.
Furthermore, we will check for target leakage. In this dataset, genuine articles almost always contain a publisher-specific string like `"Reuters"` in their dateline, whereas fake articles do not. If we don't remove this, the models will learn to look for `"Reuters"` rather than semantic attributes, resulting in shortcut learning that doesn't generalize.


In [ ]:
# Word Count Analysis
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(data=df, x='word_count', hue='label', kde=True, bins=30, palette={0: "green", 1: "red"}, multiple="stack")
plt.title('Word Count Distribution')
plt.xlabel('Number of Words')
plt.ylabel('Count')

# Target Leakage Analysis (Percentage of articles containing the word 'reuters')
reuters_count_true = df[df['label'] == 0]['text'].apply(lambda x: 'reuters' in str(x).lower()).mean()
reuters_count_fake = df[df['label'] == 1]['text'].apply(lambda x: 'reuters' in str(x).lower()).mean()

plt.subplot(1, 2, 2)
sns.barplot(x=['Real News', 'Fake News'], y=[reuters_count_true, reuters_count_fake], palette=['green', 'red'])
plt.title('Percentage of Articles Containing "Reuters"')
plt.ylabel('Ratio')
plt.ylim(0, 1.1)
for idx, val in enumerate([reuters_count_true, reuters_count_fake]):
    plt.text(idx, val + 0.02, f"{val:.2%}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


## Section 2: Text Preprocessing Pipeline
To handle target leakage and scrub the text, our custom pipeline executes:
1. **Publisher Cleansing (`strip_datelines_and_leakage`)**: Uses regular expressions to match and strip agency datelines (e.g. `WASHINGTON (Reuters) -`) and purges any occurrences of `"reuters"`.
2. **Text Cleansing**: Lowercases the text, strips HTML tags, removes URLs, replaces punctuation with single spaces, and normalizes whitespace.
3. **Stopword Filtering**: Removes trivial words using a local set of 150+ standard English stopwords to avoid external downloads.


In [ ]:
STOPWORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', 'arent', 'as', 'at',
    'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'cant', 'cannot', 'could',
    'couldnt', 'did', 'didnt', 'do', 'does', 'doesnt', 'doing', 'dont', 'down', 'during', 'each', 'few', 'for', 'from',
    'further', 'had', 'hadnt', 'has', 'hasnt', 'have', 'havent', 'having', 'he', 'hed', 'hell', 'hes', 'her', 'here',
    'heres', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'hows', 'i', 'id', 'ill', 'im', 'ive', 'if', 'in',
    'into', 'is', 'isnt', 'it', 'its', 'itself', 'lets', 'me', 'more', 'most', 'mustnt', 'my', 'myself', 'no', 'nor',
    'not', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'ought', 'our', 'ours', 'ourselves', 'out', 'over', 'own',
    'same', 'shant', 'she', 'shed', 'shell', 'shes', 'should', 'shouldnt', 'so', 'some', 'such', 'than', 'that',
    'thats', 'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'theres', 'these', 'they', 'theyd',
    'theyll', 'theyre', 'theyve', 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 'was',
    'wasnt', 'we', 'wed', 'well', 'were', 'weve', 'werent', 'what', 'whats', 'when', 'whens', 'where', 'wheres',
    'which', 'while', 'who', 'whos', 'whom', 'why', 'whys', 'with', 'wont', 'would', 'wouldnt', 'you', 'youd',
    'youll', 'youre', 'youve', 'your', 'yours', 'yourself', 'yourselves'
}

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)  # Remove HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)  # Replace punctuation with space
    text = re.sub(r'\s+', ' ', text).strip()  # Normalize whitespace
    return text

def tokenize(text):
    return text.split()

def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOPWORDS]

def strip_datelines_and_leakage(text):
    if not isinstance(text, str):
        return ""
        
    # Strip patterns like 'WASHINGTON (Reuters) - ' or 'SEOUL/LONDON (Reuters) -- '
    text_clean = re.sub(r'^[A-Z\s,\./]+ \([A-Za-z\s]+\)\s*-\s*-?', '', text)
    
    # Remove standalone reuters mentions
    text_clean = re.sub(r'\b(reuters|reuters.com)\b', '', text_clean, flags=re.IGNORECASE)
    return text_clean

def preprocess_pipeline(text):
    scrubbed = strip_datelines_and_leakage(text)
    cleaned = clean_text(scrubbed)
    tokens = tokenize(cleaned)
    filtered = remove_stopwords(tokens)
    return " ".join(filtered)

# Test Pipeline
sample = "WASHINGTON (Reuters) - The U.S. government announced new regulations today. Visit https://example.com for more."
print("Raw text:", sample)
print("Processed:", preprocess_pipeline(sample))


## Section 3: Feature Extraction (TF-IDF Vectorizer from Scratch)
To represent textual data numerically, we implement `CustomCountVectorizer` and `CustomTfidfVectorizer` from scratch.
The formulas used are:

$$\text{TF}(t, d) = \text{count of word } t \text{ in document } d$$
$$\text{IDF}(t) = \log\left(\frac{1 + N}{1 + \text{DF}(t)}\right) + 1$$
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Finally, we apply **L2 Normalization** to normalize row vectors:
$$v_{\text{norm}} = \frac{v}{\|v\|_2}$$


In [ ]:
class CustomCountVectorizer:
    def __init__(self, max_features=1000):
        self.max_features = max_features
        self.vocabulary_ = {}
        self.feature_names_ = []

    def fit(self, raw_documents):
        word_counts = Counter()
        for doc in raw_documents:
            tokens = doc.split() if isinstance(doc, str) else list(doc)
            word_counts.update(tokens)

        most_common = word_counts.most_common(self.max_features)
        self.vocabulary_ = {word: idx for idx, (word, _) in enumerate(most_common)}
        self.feature_names_ = [word for word, _ in most_common]
        return self

    def transform(self, raw_documents):
        num_docs = len(raw_documents)
        vocab_size = len(self.vocabulary_)
        X = np.zeros((num_docs, vocab_size), dtype=np.float32)

        for idx, doc in enumerate(raw_documents):
            tokens = doc.split() if isinstance(doc, str) else list(doc)
            doc_counts = Counter(tokens)
            for word, count in doc_counts.items():
                if word in self.vocabulary_:
                    X[idx, self.vocabulary_[word]] = count
        return X

    def fit_transform(self, raw_documents):
        return self.fit(raw_documents).transform(raw_documents)


class CustomTfidfVectorizer:
    def __init__(self, max_features=1000, norm='l2'):
        self.max_features = max_features
        self.norm = norm
        self.count_vectorizer = CustomCountVectorizer(max_features=max_features)
        self.idf_ = None

    def fit(self, raw_documents):
        X_counts = self.count_vectorizer.fit(raw_documents).transform(raw_documents)
        n_samples = X_counts.shape[0]
        df = np.sum(X_counts > 0, axis=0)
        
        # Smooth IDF calculation
        self.idf_ = np.log((1 + n_samples) / (1 + df)) + 1
        self.vocabulary_ = self.count_vectorizer.vocabulary_
        self.feature_names_ = self.count_vectorizer.feature_names_
        return self

    def transform(self, raw_documents):
        X_counts = self.count_vectorizer.transform(raw_documents)
        X_tfidf = X_counts * self.idf_

        if self.norm == 'l2':
            norms = np.linalg.norm(X_tfidf, axis=1, keepdims=True)
            norms[norms == 0.0] = 1.0
            X_tfidf = X_tfidf / norms
        return X_tfidf

    def fit_transform(self, raw_documents):
        return self.fit(raw_documents).transform(raw_documents)


## Section 4: Model Evaluation Utilities
To validate model accuracy, precision, recall, and F1-score, we implement standard evaluation formulas:
$$\text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$
$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

We also write a randomized `train_test_split_scratch` to slice data indices randomly.


In [ ]:
def train_test_split_scratch(X, y, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
        
    n_samples = len(X)
    shuffled_indices = np.random.permutation(n_samples)
    test_set_size = int(n_samples * test_size)
    
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    
    if isinstance(X, np.ndarray):
        X_train, X_test = X[train_indices], X[test_indices]
    else:
        X_train = [X[i] for i in train_indices]
        X_test = [X[i] for i in test_indices]
        
    if isinstance(y, np.ndarray):
        y_train, y_test = y[train_indices], y[test_indices]
    else:
        y_train = [y[i] for i in train_indices]
        y_test = [y[i] for i in test_indices]
        
    return X_train, X_test, y_train, y_test

def confusion_matrix_scratch(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    matrix = [
        [TN, FP],
        [FN, TP]
    ]
    return TP, FP, TN, FN, matrix

def accuracy_score_scratch(y_true, y_pred):
    return np.mean(np.array(y_true) == np.array(y_pred))

def precision_score_scratch(y_true, y_pred):
    TP, FP, _, _, _ = confusion_matrix_scratch(y_true, y_pred)
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0

def recall_score_scratch(y_true, y_pred):
    TP, _, _, FN, _ = confusion_matrix_scratch(y_true, y_pred)
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0

def f1_score_scratch(y_true, y_pred):
    prec = precision_score_scratch(y_true, y_pred)
    rec = recall_score_scratch(y_true, y_pred)
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def classification_report_scratch(y_true, y_pred):
    accuracy = accuracy_score_scratch(y_true, y_pred)
    precision = precision_score_scratch(y_true, y_pred)
    recall = recall_score_scratch(y_true, y_pred)
    f1 = f1_score_scratch(y_true, y_pred)
    TP, FP, TN, FN, _ = confusion_matrix_scratch(y_true, y_pred)
    
    return (
        f"Classification Metrics Report:\n"
        f"-------------------------------\n"
        f"Accuracy : {accuracy:.4f}\n"
        f"Precision: {precision:.4f}\n"
        f"Recall   : {recall:.4f}\n"
        f"F1-Score : {f1:.4f}\n\n"
        f"Confusion Matrix Details:\n"
        f"-----------------------\n"
        f"True Positives (TP) : {TP}\n"
        f"False Positives (FP): {FP}\n"
        f"True Negatives (TN) : {TN}\n"
        f"False Negatives (FN): {FN}\n"
    )


## Section 5: Machine Learning Models from Scratch
Here, we implement the 4 core models that will classify our vectorized news texts.
1. **BaseModel**: Standard interface enforcing `fit` and `predict` behaviors.
2. **KNNClassifier**: Performs voting based on closest examples. We support **Cosine Distance** since it evaluates vector directionality rather than magnitude, which is mathematically superior for high-dimensional sparse TF-IDF vectors:
   $$D_{\text{cosine}}(u, v) = 1 - \frac{u \cdot v}{\|u\|_2 \|v\|_2}$$
3. **LogisticRegressionClassifier**: Uses mini-batch gradient descent and includes L2 regularization:
   $$w \leftarrow w - \eta \left( \frac{1}{m} X^T (\hat{y} - y) + \frac{\lambda}{m} w \right)$$
4. **SimpleNeuralNetwork**: A Feedforward Multi-Layer Perceptron (Input $\rightarrow$ Hidden (ReLU) $\rightarrow$ Output (Sigmoid)) optimized via mini-batch SGD.
5. **RandomForestClassifier**: Ensemble of Decision Trees with node splits selected using Gini Impurity Information Gain.


In [ ]:
class BaseModel:
    def fit(self, X, y):
        raise NotImplementedError("Each model must implement a fit method.")
    def predict(self, X):
        raise NotImplementedError("Each model must implement a predict method.")
    def score(self, X, y):
        predictions = self.predict(X)
        return np.mean(predictions == y)

class KNNClassifier(BaseModel):
    def __init__(self, k=5, metric='cosine'):
        self.k = k
        self.metric = metric
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self

    def predict(self, X):
        X = np.array(X)
        predictions = []
        batch_size = 500
        n_samples = X.shape[0]
        
        for i in range(0, n_samples, batch_size):
            X_batch = X[i : i + batch_size]
            if self.metric == 'cosine':
                norm_batch = np.linalg.norm(X_batch, axis=1, keepdims=True)
                norm_train = np.linalg.norm(self.X_train, axis=1, keepdims=True).T
                norm_batch[norm_batch == 0] = 1e-8
                norm_train[norm_train == 0] = 1e-8
                dot_product = np.dot(X_batch, self.X_train.T)
                dists = 1.0 - (dot_product / (norm_batch * norm_train))
            else:
                x2 = np.sum(X_batch ** 2, axis=1, keepdims=True)
                y2 = np.sum(self.X_train ** 2, axis=1, keepdims=True).T
                xy = np.dot(X_batch, self.X_train.T)
                dists = np.sqrt(np.clip(x2 + y2 - 2 * xy, 0, None))
                
            k_nearest_indices = np.argpartition(dists, self.k - 1, axis=1)[:, :self.k]
            
            for row_idx, indices in enumerate(k_nearest_indices):
                row_dists = dists[row_idx, indices]
                sorted_k_indices = indices[np.argsort(row_dists)]
                neighbor_labels = self.y_train[sorted_k_indices]
                counts = np.bincount(neighbor_labels.astype(int))
                predictions.append(np.argmax(counts))
                
        return np.array(predictions)


class LogisticRegressionClassifier(BaseModel):
    def __init__(self, lr=0.1, epochs=100, lambda_reg=0.01, batch_size=None, verbose=False):
        self.lr = lr
        self.epochs = epochs
        self.lambda_reg = lambda_reg
        self.batch_size = batch_size
        self.verbose = verbose
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        n_samples, n_features = X.shape
        self.weights = np.zeros((n_features, 1))
        self.bias = 0.0
        self.loss_history = []
        
        for epoch in range(self.epochs):
            if self.batch_size is not None and self.batch_size < n_samples:
                indices = np.arange(n_samples)
                np.random.shuffle(indices)
                X_shff = X[indices]
                y_shff = y[indices]
                epoch_loss = 0.0
                num_batches = int(np.ceil(n_samples / self.batch_size))
                
                for b in range(num_batches):
                    s_idx = b * self.batch_size
                    e_idx = min(s_idx + self.batch_size, n_samples)
                    X_b, y_b = X_shff[s_idx:e_idx], y_shff[s_idx:e_idx]
                    m_b = X_b.shape[0]
                    
                    a = self._sigmoid(np.dot(X_b, self.weights) + self.bias)
                    loss = -np.mean(y_b * np.log(a + 1e-15) + (1 - y_b) * np.log(1 - a + 1e-15))
                    reg = (self.lambda_reg / (2 * m_b)) * np.sum(self.weights ** 2)
                    epoch_loss += (loss + reg) * (m_b / n_samples)
                    
                    dz = a - y_b
                    dw = (1 / m_b) * np.dot(X_b.T, dz) + (self.lambda_reg / m_b) * self.weights
                    db = (1 / m_b) * np.sum(dz)
                    
                    self.weights -= self.lr * dw
                    self.bias -= self.lr * db
                self.loss_history.append(epoch_loss)
            else:
                a = self._sigmoid(np.dot(X, self.weights) + self.bias)
                loss = -np.mean(y * np.log(a + 1e-15) + (1 - y) * np.log(1 - a + 1e-15))
                reg = (self.lambda_reg / (2 * n_samples)) * np.sum(self.weights ** 2)
                self.loss_history.append(loss + reg)
                
                dz = a - y
                dw = (1 / n_samples) * np.dot(X.T, dz) + (self.lambda_reg / n_samples) * self.weights
                db = (1 / n_samples) * np.sum(dz)
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
                
        return self

    def predict_proba(self, X):
        return self._sigmoid(np.dot(np.array(X), self.weights) + self.bias).flatten()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def partial_fit(self, X, y, lr=None):
        if self.weights is None:
            self.weights = np.zeros((X.shape[1], 1))
            self.bias = 0.0
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        step_lr = lr if lr is not None else self.lr
        z = np.dot(X, self.weights) + self.bias
        y_pred = self._sigmoid(z)
        error = y_pred - y
        m = X.shape[0]
        dw = (1 / m) * np.dot(X.T, error) + (self.lambda_reg / m) * self.weights
        db = (1 / m) * np.sum(error)
        self.weights -= step_lr * dw
        self.bias -= step_lr * db
        return self


In [ ]:
class SimpleNeuralNetwork(BaseModel):
    def __init__(self, hidden_dim=64, lr=0.01, epochs=100, batch_size=64, verbose=False):
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose
        self.W1, self.b1, self.W2, self.b2 = None, None, None, None
        self.loss_history = []

    def _relu(self, z):
        return np.maximum(0, z)

    def _relu_derivative(self, z):
        return (z > 0).astype(float)

    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        n_samples, n_features = X.shape
        
        # He Initialization
        np.random.seed(42)
        self.W1 = np.random.randn(n_features, self.hidden_dim) * np.sqrt(2.0 / n_features)
        self.b1 = np.zeros((1, self.hidden_dim))
        self.W2 = np.random.randn(self.hidden_dim, 1) * np.sqrt(2.0 / self.hidden_dim)
        self.b2 = np.zeros((1, 1))
        self.loss_history = []
        
        for epoch in range(self.epochs):
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            X_shff = X[indices]
            y_shff = y[indices]
            epoch_loss = 0.0
            num_batches = int(np.ceil(n_samples / self.batch_size))
            
            for b in range(num_batches):
                s_idx = b * self.batch_size
                e_idx = min(s_idx + self.batch_size, n_samples)
                X_b, y_b = X_shff[s_idx:e_idx], y_shff[s_idx:e_idx]
                m_b = X_b.shape[0]
                
                # Forward Pass
                Z1 = np.dot(X_b, self.W1) + self.b1
                A1 = self._relu(Z1)
                Z2 = np.dot(A1, self.W2) + self.b2
                A2 = self._sigmoid(Z2)
                
                loss = -np.mean(y_b * np.log(A2 + 1e-15) + (1 - y_b) * np.log(1 - A2 + 1e-15))
                epoch_loss += loss * (m_b / n_samples)
                
                # Backpropagation
                dZ2 = A2 - y_b
                dW2 = (1.0 / m_b) * np.dot(A1.T, dZ2)
                db2 = (1.0 / m_b) * np.sum(dZ2, axis=0, keepdims=True)
                
                dZ1 = np.dot(dZ2, self.W2.T) * self._relu_derivative(Z1)
                dW1 = (1.0 / m_b) * np.dot(X_b.T, dZ1)
                db1 = (1.0 / m_b) * np.sum(dZ1, axis=0, keepdims=True)
                
                # Param updates
                self.W1 -= self.lr * dW1
                self.b1 -= self.lr * db1
                self.W2 -= self.lr * dW2
                self.b2 -= self.lr * db2
                
            self.loss_history.append(epoch_loss)
        return self

    def predict_proba(self, X):
        X = np.array(X)
        Z1 = np.dot(X, self.W1) + self.b1
        A1 = self._relu(Z1)
        return self._sigmoid(np.dot(A1, self.W2) + self.b2).flatten()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def partial_fit(self, X, y, lr=None):
        if self.W1 is None:
            n_features = X.shape[1]
            self.W1 = np.random.randn(n_features, self.hidden_dim) * np.sqrt(2.0 / n_features)
            self.b1 = np.zeros((1, self.hidden_dim))
            self.W2 = np.random.randn(self.hidden_dim, 1) * np.sqrt(2.0 / self.hidden_dim)
            self.b2 = np.zeros((1, 1))
            
        X = np.array(X)
        y = np.array(y).reshape(-1, 1)
        step_lr = lr if lr is not None else self.lr
        Z1 = np.dot(X, self.W1) + self.b1
        A1 = self._relu(Z1)
        Z2 = np.dot(A1, self.W2) + self.b2
        A2 = self._sigmoid(Z2)
        m = X.shape[0]
        dZ2 = A2 - y
        dW2 = (1 / m) * np.dot(A1.T, dZ2)
        db2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self._relu_derivative(Z1)
        dW1 = (1 / m) * np.dot(X.T, dZ1)
        db1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)
        self.W1 -= step_lr * dW1
        self.b1 -= step_lr * db1
        self.W2 -= step_lr * dW2
        self.b2 -= step_lr * db2
        return self


class DecisionNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
    def is_leaf(self):
        return self.value is not None

class DecisionTreeClassifier(BaseModel):
    def __init__(self, max_depth=10, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None

    def _gini(self, y):
        m = len(y)
        if m == 0: return 0.0
        counts = np.bincount(y)
        probs = counts / m
        return 1.0 - np.sum(probs ** 2)

    def _best_split(self, X, y, feature_indices):
        best_gain = -1.0
        split_idx, split_thresh = None, None
        n_samples = X.shape[0]
        current_gini = self._gini(y)
        
        for feat_idx in feature_indices:
            X_column = X[:, feat_idx]
            uniques = np.unique(X_column)
            if len(uniques) > 10:
                thresholds = np.percentile(X_column, [10, 30, 50, 70, 90])
            else:
                thresholds = uniques
                
            for thresh in thresholds:
                left_mask = X_column <= thresh
                right_mask = ~left_mask
                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0: continue
                y_left, y_right = y[left_mask], y[right_mask]
                w_left = len(y_left) / n_samples
                w_right = len(y_right) / n_samples
                gain = current_gini - (w_left * self._gini(y_left) + w_right * self._gini(y_right))
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh
        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = np.argmax(np.bincount(y)) if len(y) > 0 else 0
            return DecisionNode(value=leaf_value)
            
        if self.max_features is None:
            feature_indices = np.arange(n_features)
        else:
            max_feat = min(n_features, self.max_features)
            feature_indices = np.random.choice(n_features, max_feat, replace=False)
            
        best_feat, best_thresh = self._best_split(X, y, feature_indices)
        if best_feat is None:
            leaf_value = np.argmax(np.bincount(y)) if len(y) > 0 else 0
            return DecisionNode(value=leaf_value)
            
        left_mask = X[:, best_feat] <= best_thresh
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        return DecisionNode(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).astype(int)
        self.root = self._build_tree(X, y)
        return self

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._traverse_tree(x, self.root) for x in X])

class RandomForestClassifier(BaseModel):
    def __init__(self, n_estimators=10, max_depth=8, min_samples_split=2, max_features='sqrt'):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y).astype(int)
        n_samples, n_features = X.shape
        self.trees = []
        
        if self.max_features == 'sqrt':
            max_feats = int(np.sqrt(n_features))
        elif isinstance(self.max_features, int):
            max_feats = self.max_features
        else:
            max_feats = n_features
            
        for _ in range(self.n_estimators):
            bootstrap_idx = np.random.choice(n_samples, n_samples, replace=True)
            X_bootstrap = X[bootstrap_idx]
            y_bootstrap = y[bootstrap_idx]
            tree = DecisionTreeClassifier(max_depth=self.max_depth, min_samples_split=self.min_samples_split, max_features=max_feats)
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)
        return self

    def predict(self, X):
        X = np.array(X)
        tree_preds = np.array([tree.predict(X) for tree in self.trees]).T
        final_preds = []
        for sample_preds in tree_preds:
            counts = np.bincount(sample_preds)
            final_preds.append(np.argmax(counts))
        return np.array(final_preds)


## Section 6: Pipeline Execution, Evaluation, and Plotting
Now we run the complete classification pipeline. We will:
1. Extract cleaned strings for all articles.
2. Perform a TF-IDF transform with a maximum of 1,000 features.
3. Divide the dataset into an 80/20 train/test split.
4. Train each model, tracking execution times.
5. Print evaluation summaries for each model.
6. Plot loss curves for the iterative learning algorithms (Logistic Regression and Neural Network).


In [ ]:
texts = df['text'].fillna("").values
labels = df['label'].values

print("1. Preprocessing texts...")
start = time.time()
clean_texts = [preprocess_pipeline(t) for t in texts]
print(f"   Processed {len(clean_texts)} texts in {time.time() - start:.2f} seconds.")

vocab_size = 1000
print(f"\n2. Transforming texts using Custom TF-IDF (max_features={vocab_size})...")
start = time.time()
vectorizer = CustomTfidfVectorizer(max_features=vocab_size)
X = vectorizer.fit_transform(clean_texts)
print(f"   Feature matrix shape: {X.shape} (Time: {time.time() - start:.2f}s)")

print("\n3. Splitting into Train and Test subsets...")
X_train, X_test, y_train, y_test = train_test_split_scratch(X, labels, test_size=0.2, random_state=42)
print(f"   Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

print("\n4. Training models...")
models = {
    "K-Nearest Neighbors": KNNClassifier(k=5, metric='cosine'),
    "Logistic Regression": LogisticRegressionClassifier(lr=0.1, epochs=150, lambda_reg=0.01, verbose=False),
    "Random Forest": RandomForestClassifier(n_estimators=10, max_depth=8),
    "Neural Network": SimpleNeuralNetwork(hidden_dim=64, lr=0.05, epochs=100, batch_size=32)
}

results = {}
for name, model in models.items():
    print(f"   Training {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    print(f"      Done in {train_time:.2f} seconds.")
    
    preds = model.predict(X_test)
    acc = accuracy_score_scratch(y_test, preds)
    results[name] = {
        'model': model,
        'accuracy': acc,
        'preds': preds,
        'time': train_time
    }
    print(f"      Accuracy: {acc:.2%}")

print("\n" + "="*60)
print(f"{'Model Name':25s} | {'Test Accuracy':13s} | {'Train Time':12s}")
print("="*60)
for name, res in results.items():
    print(f"{name:25s} | {res['accuracy']:13.2%} | {res['time']:10.2f}s")
print("="*60)


### Convergence Analysis (Loss Curves)
Let's review the loss history for Logistic Regression (L2-Regularized Cross-Entropy) and our Feedforward Neural Network (Binary Cross-Entropy over epochs).


In [ ]:
plt.figure(figsize=(15, 6))

# Logistic Regression Loss Curve
plt.subplot(1, 2, 1)
plt.plot(results["Logistic Regression"]['model'].loss_history, color='blue', lw=2)
plt.title('Logistic Regression Convergence Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss Value (Regularized)')
plt.grid(True)

# Neural Network Loss Curve
plt.subplot(1, 2, 2)
plt.plot(results["Neural Network"]['model'].loss_history, color='orange', lw=2)
plt.title('Neural Network Convergence Curve')
plt.xlabel('Epochs')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True)

plt.tight_layout()
plt.show()


## Section 7: Detailed Classification Reports & Confusion Matrices
For each classifier, let's output a full classification report (Accuracy, Precision, Recall, F1) and render the Confusion Matrix as a Seaborn heatmap.


In [ ]:
for name, res in results.items():
    print(f"\n==================== {name} ====================")
    print(classification_report_scratch(y_test, res['preds']))
    
    TP, FP, TN, FN, matrix = confusion_matrix_scratch(y_test, res['preds'])
    
    # Plot Confusion Matrix Heatmap
    plt.figure(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Real (0)', 'Fake (1)'], 
                yticklabels=['Real (0)', 'Fake (1)'])
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()


## Section 8: Live Article Verification Console
Use the function below to check custom headlines. Text will run through the exact preprocessing and feature extraction vectors, then get evaluated by all 4 models.


In [ ]:
def predict_custom_text(user_text):
    print(f"Original Article Content:\n\"{{user_text}}\"\n")
    cleaned = preprocess_pipeline(user_text)
    x_input = vectorizer.transform([cleaned])
    
    print(f"{'Classifier Name':25s} | {'Verdict'}")
    print("-" * 40)
    for name, res in results.items():
        pred = res['model'].predict(x_input)[0]
        verdict = "🔴 FAKE NEWS" if pred == 1 else "🟢 REAL NEWS"
        print(f"{name:25s} | {verdict}")

# Run custom predictions
sample_headline = "BREAKING: NASA scientists announce they have discovered signs of active civilization on Mars."
predict_custom_text(sample_headline)
